# Benchmark Forward — Simulador Python EM 1D TIV

**Geosteering AI v2.0** — Sprint 2.7 (Gate final Fase 2)

Este notebook executa o benchmark do simulador Python otimizado no Google Colab,
medindo o throughput em modelos/hora e comparando contra:

- **Meta Python Numba**: ≥ 40 000 mod/h (CPU)
- **Baseline Fortran OpenMP**: 58 856 mod/h (8 cores)

## Instruções

1. Conecte ao runtime Colab (CPU ou GPU)
2. Execute todas as células em sequência
3. Os resultados aparecem na última célula como tabela markdown

**Requisitos**: NumPy, Numba (instalados automaticamente abaixo)

In [ ]:
# ── Célula 1: Instalar dependências e clonar o repositório ──────
import subprocess
import sys
import os

# Instalar numba se não disponível
try:
    import numba
    print(f'Numba já instalado: {numba.__version__}')
except ImportError:
    print('Instalando numba...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numba'])
    import numba
    print(f'Numba instalado: {numba.__version__}')

# Clonar o repositório se não existir
REPO_DIR = '/content/geosteering-ai'
if not os.path.exists(REPO_DIR):
    print('Clonando repositório...')
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/daniel-guitarplayer-8/geosteering-ai.git',
        REPO_DIR
    ])
else:
    print(f'Repositório já existe em {REPO_DIR}')
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])

# Adicionar ao path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('\n--- Setup concluído ---')

In [ ]:
# ── Célula 2: Verificar imports e Numba ──────────────────────────
import numpy as np
from geosteering_ai.simulation import simulate, SimulationConfig
from geosteering_ai.simulation._numba.propagation import HAS_NUMBA
from geosteering_ai.simulation.benchmarks.bench_forward import (
    run_benchmark, run_all_sizes, BenchmarkResult,
    FORTRAN_BASELINE_MOD_PER_HOUR, PYTHON_TARGET_MOD_PER_HOUR,
)
import platform, os

print(f'NumPy: {np.__version__}')
print(f'Numba disponível: {HAS_NUMBA}')
print(f'Plataforma: {platform.system()} {platform.machine()}')
print(f'CPU cores: {os.cpu_count()}')
print(f'Python: {platform.python_version()}')
print(f'\nMeta: ≥ {PYTHON_TARGET_MOD_PER_HOUR:,} mod/h')
print(f'Baseline Fortran: {FORTRAN_BASELINE_MOD_PER_HOUR:,} mod/h')

In [ ]:
# ── Célula 3: Smoke-test rápido (validação antes do benchmark) ───
result = simulate(
    rho_h=np.array([100.0]),
    rho_v=np.array([100.0]),
    esp=np.zeros(0, dtype=np.float64),
    positions_z=np.linspace(-5, 5, 20),
    frequency_hz=20000.0,
    tr_spacing_m=1.0,
)
assert result.H_tensor.shape == (20, 1, 9)
assert np.all(np.isfinite(result.H_tensor))
print(f'Smoke-test PASS: shape={result.H_tensor.shape}, all finite')
print(f'Hzz[10] = {result.H_tensor[10, 0, 8]}')

In [ ]:
# ── Célula 4: Benchmark SMALL (3 camadas, 100 posições) ──────────
print('Rodando benchmark SMALL...')
r_small = run_benchmark(size='small', n_iter=20, n_warmup=5)
print(r_small.to_markdown())

In [ ]:
# ── Célula 5: Benchmark MEDIUM (7 camadas TIV, 300 posições) ─────
print('Rodando benchmark MEDIUM...')
r_medium = run_benchmark(size='medium', n_iter=20, n_warmup=5)
print(r_medium.to_markdown())

In [ ]:
# ── Célula 6: Benchmark LARGE (22 camadas TIV Sobol, 601 posições)
print('Rodando benchmark LARGE (pode levar ~1 min)...')
r_large = run_benchmark(size='large', n_iter=10, n_warmup=3)
print(r_large.to_markdown())

In [ ]:
# ── Célula 7: Resumo comparativo ─────────────────────────────────
import pandas as pd

summary = pd.DataFrame([
    {
        'Modelo': r.model_name,
        'Camadas': r.n_layers,
        'Posições': r.n_positions,
        'Tempo médio (s)': f'{r.mean_time_s:.4f}',
        'Throughput (mod/h)': f'{r.throughput_models_per_hour:,.0f}',
        '% Fortran': f'{100*r.throughput_models_per_hour/FORTRAN_BASELINE_MOD_PER_HOUR:.1f}%',
        'Gate ≥40k': '✅' if r.pass_target else '❌',
        'Numba': '✅' if r.has_numba else '❌',
    }
    for r in [r_small, r_medium, r_large]
])

print('\n## Resumo Comparativo\n')
print(summary.to_markdown(index=False))
print(f'\nPlataforma: {r_large.platform_info}')
print(f'Baseline Fortran OpenMP: {FORTRAN_BASELINE_MOD_PER_HOUR:,} mod/h')

In [ ]:
# ── Célula 8: Gráfico de tempos por iteração (large) ─────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, r) in zip(axes, [
    ('Small', r_small), ('Medium', r_medium), ('Large', r_large)
]):
    ax.bar(range(1, len(r.times_seconds)+1), r.times_seconds, color='steelblue')
    ax.axhline(r.mean_time_s, color='red', linestyle='--', label=f'Média: {r.mean_time_s:.4f}s')
    ax.set_title(f'{name} ({r.throughput_models_per_hour:,.0f} mod/h)')
    ax.set_xlabel('Iteração')
    ax.set_ylabel('Tempo (s)')
    ax.legend()

plt.suptitle('Benchmark Forward — Simulador Python EM 1D TIV', fontweight='bold')
plt.tight_layout()
plt.show()